In [10]:
# testing hysteresis parameter calculation? 
import numpy as np
import scipy.integrate as spi
from scipy.optimize import curve_fit
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns  

storm_directory = 'data/constituents'
storms = {}
for filename in os.listdir(storm_directory):
    # check if the file is a CSV file
    if filename.endswith('.csv'):
        file_path = os.path.join(storm_directory, filename) # construct the full file path
        df = pd.read_csv(file_path)                         # read the CSV file into a data frame
        df = df.dropna(subset=['Date_Time'])                # drop rows where 'Date/Time' is NaN  
        df['Date_Time'] = pd.to_datetime(df['Date_Time'])   # convert to datetime format
        df = df.set_index('Date_Time')                      # set date time as the index 
        df = df.dropna(how='all', axis=1)                   # drop columns where all values are NaN
        key = filename[:-4]                                 # remove the '.csv' from the filename to use as the dictionary key
        storms[key] = df                                    # store the data frame in the dictionary

shear_stress = pd.read_csv('data/shear_stress/average_shear_stress_lajara_full.csv', parse_dates=['datetime'], index_col='datetime')
sonde_downstream = pd.read_csv('data/sonde_data/sonde_down_full_record.csv', parse_dates=['DateTime'], index_col='DateTime')
sonde_upstream = pd.read_csv('data/sonde_data/sonde_up_full_record.csv', parse_dates=['DateTime'], index_col='DateTime')


In [7]:
storms['st1_down']

,SS (uL/L),SSC (mg/L),SRP (mg/L),TP (mg/L),PP (mg/L),POC (mg/L),DOC (mg/L)
Date_Time,,,,,,,
2021-07-23 14:30:00,0.00000,4.409565,0.02625,0.03125,0.00,2.864641,1.754
2021-07-23 15:50:00,339.78248,562.325800,0.06200,0.36000,1.19,51.660000,9.180
2021-07-23 16:09:00,457.12000,528.034800,0.08800,0.44500,1.63,50.228000,12.630
2021-07-23 16:40:00,152.79000,143.169000,0.08800,0.24800,0.77,14.502000,13.390
2021-07-23 16:55:00,168.45000,82.761400,0.08300,0.21400,0.50,11.681000,13.000
2021-07-23 19:19:00,43.15000,36.920000,0.05300,0.09700,0.10,6.411000,12.870
2021-07-23 21:02:00,17.70000,22.011900,0.04700,0.07700,0.07,4.805000,11.550


Process and Merge Data

In [ ]:
# Merge each storm with shear stress and sonde variables on matching timestamps
merged_storms = {}

for key, df in storms.items():
    if "down" in key.lower():
        sonde_df = sonde_downstream
    elif "up" in key.lower():
        sonde_df = sonde_upstream
    else:
        continue

    matching_times = df.index.intersection(shear_stress.index).intersection(sonde_df.index)

    merged_storms[key] = (
        df.loc[matching_times]
        .join(shear_stress.loc[matching_times, ["shear_stress"]], how="inner")
        .join(sonde_df.loc[matching_times, ["Turbidity FNU", "fDOM RFU"]], how="inner")
    )

# Example:
# merged_storms['st1_down']
